In [ ]:
!pip install gym==0.26.2

In [ ]:
import numpy as np
import gym

env = gym.make('CliffWalking-v0')

# ----------------- 需要补全的部分 1：初始化 Q 表格 -----------------
# 状态总数 = 48 (4行×12列)，动作数 = 4
Q = np.zeros((env.observation_space.n, env.action_space.n))
# -----------------------------------------------------------------

# 超参数（可根据实验调整）
alpha = 0.5       # 学习率
gamma = 0.99      # 折扣因子
epsilon = 0.1     # 探索概率
episodes = 500    # 训练回合数

# 用于记录每个回合的总奖励
rewards_per_episode = []

for episode in range(episodes):
    state, info = env.reset()   # 重置环境，获取初始状态 s
    total_reward = 0
    done = False

    while not done:
        # ----------------- 需要补全的部分 2：ε-greedy 选择动作 -----------------
        if np.random.rand() < epsilon:
            action = env.action_space.sample()      # 随机探索
        else:
            action = np.argmax(Q[state, :])         # 利用当前最优策略
        # ---------------------------------------------------------------------

        # 执行动作，观察下一状态、奖励、是否终止
        next_state, reward, done, truncated, info = env.step(action)
        
        # ----------------- 需要补全的部分 3：Q 值更新公式 ----------------------
        # Q(s,a) ← Q(s,a) + α [ r + γ·max_a' Q(s',a') - Q(s,a) ]
        best_next_action = np.argmax(Q[next_state, :])
        td_target = reward + gamma * Q[next_state, best_next_action]
        td_error = td_target - Q[state, action]
        Q[state, action] += alpha * td_error
        # --------------------------------------------------------------------

        state = next_state
        total_reward += reward

    rewards_per_episode.append(total_reward)

# 评估训练后的策略（关闭探索）
state, info = env.reset()
done = False
eval_reward = 0
while not done:
    action = np.argmax(Q[state, :])          # 纯贪婪策略
    state, reward, done, truncated, info = env.step(action)
    eval_reward += reward
print(f"评估回合奖励: {eval_reward} (最优值应接近 -13)")

env.close()

### 第一类：学习曲线（Average Learning Curve）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 假设已经在训练循环中记录了每个回合的总奖励
# rewards_per_episode 是一个列表，长度为 episodes

def plot_learning_curve(rewards, window_size=20):
    """
    绘制原始奖励和平滑后的奖励曲线
    :param rewards: list，每个回合的总奖励
    :param window_size: 滑动窗口大小，用于计算移动平均
    """
    plt.figure(figsize=(10, 5))
    
    episodes = np.arange(len(rewards))
    
    # 原始奖励散点（浅色）
    plt.scatter(episodes, rewards, s=2, alpha=0.3, color='blue', label='Raw Rewards')
    
    # 计算移动平均（Running Average）
    running_avg = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
    plt.plot(episodes[window_size-1:], running_avg, 
             color='red', linewidth=2, label=f'Running Average (window={window_size})')
    
    # 画一条最优参考线（-13）
    plt.axhline(y=-13, color='green', linestyle='--', label='Optimal (-13)')
    
    plt.xlabel('Episodes')
    plt.ylabel('Total Reward')
    plt.title('Average Learning Curve of Q-Learning for Maze')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 调用绘图函数
plot_learning_curve(rewards_per_episode)

### 第二类：智能体走迷宫轨迹图（热力图/路径图）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_maze_path(q_table, env):
    """
    利用训练好的 Q 表格，画出从起点到终点的最优路径
    适用于 4x12 的 CliffWalking 迷宫
    """
    # 网格定义：CliffWalking 是 4 行 12 列
    rows, cols = 4, 12
    
    # 创建颜色矩阵：0=普通路(白), 1=悬崖(黑), 2=起点(蓝), 3=终点(黄)
    maze_grid = np.zeros((rows, cols))
    
    # 悬崖区域 (CliffWalking: 最后一行，第1列到第10列是悬崖)
    maze_grid[3, 1:11] = 1
    
    # 起点 (左下角: 第3行，第0列)
    start = (3, 0)
    # 终点 (右下角: 第3行，第11列)
    goal = (3, 11)
    
    # 获取最优路径
    state = env.reset()[0]  # Gymnasium 返回 (obs, info)
    path = [start]
    
    done = False
    while not done:
        action = np.argmax(q_table[state, :])
        next_state, reward, done, truncated, info = env.step(action)
        # 将状态索引转换为行列坐标
        row = next_state // 12
        col = next_state % 12
        path.append((row, col))
        state = next_state
        if done:
            break
    
    # 绘制网格
    fig, ax = plt.subplots(figsize=(8, 3))
    
    # 背景色：白色为普通路，黑色为悬崖
    cmap = plt.cm.colors.ListedColormap(['white', 'black', 'lightblue', 'yellow'])
    ax.imshow(maze_grid, cmap=cmap, aspect='equal')
    
    # 标记起点和终点
    ax.text(start[1], start[0], 'S', ha='center', va='center', fontweight='bold', color='blue')
    ax.text(goal[1], goal[0], 'G', ha='center', va='center', fontweight='bold', color='red')
    
    # 绘制路径箭头
    path = np.array(path)
    ax.plot(path[:, 1], path[:, 0], marker='o', color='red', linewidth=2, markersize=4)
    
    # 在每个格子上加箭头表示动作方向（可选）
    for i in range(len(path)-1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        ax.arrow(c1, r1, c2-c1, r2-r1, head_width=0.2, head_length=0.2, fc='blue', ec='blue')
    
    # 设置刻度
    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.grid(True, color='gray', linewidth=0.5)
    
    plt.title('Optimal Path Found by Q-Learning')
    plt.show()

# 调用绘图
plot_maze_path(Q, env)

### 第三类：Q 值热力图（可选进阶）

In [ ]:
%matplotlib inline
def plot_q_values_heatmap(q_table):
    """绘制每个状态的最大 Q 值分布（Value Function）"""
    values = np.max(q_table, axis=1).reshape(4, 12)
    plt.figure(figsize=(10, 4))
    plt.imshow(values, cmap='viridis', aspect='equal')
    plt.colorbar(label='Max Q-value')
    plt.title('State Value Function (V*) Learned by Q-Learning')
    plt.xticks(range(12))
    plt.yticks(range(4))
    plt.grid(False)
    plt.show()

plot_q_values_heatmap(Q)
    